# Rotary Position Embedding（RoPE）

源码导航：[`core/position/rope.py`](../../../core/position/rope.py) 中的 `compute_rope_freqs`、`apply_rope`、`RotaryPositionalEmbedding`。

## 1. 理论推导

### 1.1 为什么需要 RoPE？

GPT-2 使用 Absolute Learned Position Embedding：将位置向量 $e_t$ 直接加到 token embedding 上。这类方案有两个已知问题：
1. **无法外推**：对超过训练长度 $T_\text{max}$ 的序列，位置 $e_{T_\text{max}+k}$ 在训练中从未出现，行为未定义。
2. **绝对位置偏差**：注意力权重依赖绝对位置，相同的两个 token 在不同绝对位置处会得到不同的注意力分数，而语义上相对位置更重要。

Su et al. (2021) 提出 **RoPE**：不修改 token embedding，而是在 attention 的 Q/K 投影后施加旋转变换，使 Q/K 点积**仅依赖相对位置**。

### 1.2 旋转公式推导

对于 head_dim 维度的 Q/K 向量，每两个相邻分量 $(x_{2i},\, x_{2i+1})$ 构成一个二维子空间，在位置 $t$ 处施加旋转角 $t \cdot \omega_i$：

$$\omega_i = \theta^{-2i/d_h}, \quad i = 0, 1, \ldots, \frac{d_h}{2} - 1, \quad \theta = 10^6 \text{（Walkie 默认值）}$$

**旋转矩阵形式**（精确定义）：

$$\begin{bmatrix} x'_{2i} \\ x'_{2i+1} \end{bmatrix} = \begin{bmatrix} \cos(t\omega_i) & -\sin(t\omega_i) \\ \sin(t\omega_i) & \cos(t\omega_i) \end{bmatrix} \begin{bmatrix} x_{2i} \\ x_{2i+1} \end{bmatrix}$$

**批量实现形式**（在实际代码中使用的等价公式）：

将向量 $x \in \mathbb{R}^{d_h}$ 拆成前半 $x_1 = x[:\frac{d}{2}]$ 和后半 $x_2 = x[\frac{d}{2}:]$，则旋转等价于：

$$\text{RoPE}(x, t) = x_1 \cdot \cos(t\Omega) - x_2 \cdot \sin(t\Omega) \;\||\; x_2 \cdot \cos(t\Omega) + x_1 \cdot \sin(t\Omega)$$

其中 $\Omega = [\omega_0, \omega_1, \ldots, \omega_{d/2-1}]$ 为频率向量，$\|$ 表示拼接。

**关键性质**：位置 $s$ 的 Q 与位置 $t$ 的 K 的点积满足：
$$\langle \text{RoPE}(q, s),\, \text{RoPE}(k, t) \rangle = f(q, k, s-t)$$

即点积只依赖 $q, k$ 的内容和**相对位置** $s - t$，不依赖绝对位置。

### 1.3 Walkie 的 $\theta = 10^6$

LLaMA/GPT-NeoX 使用 $\theta = 10^4$，Walkie 使用 $\theta = 10^6$（YaRN/LongRoPE 等长序列工作的结论）。较大的 $\theta$ 使所有频率 $\omega_i$ 整体偏小，低维度子空间旋转得更慢，外推到更长序列时高频混叠问题减轻。

### 1.4 共享位置编码：Why / What / How

**What**：在 Walkie 的 GQA 中，$H$ 个 Q 头和 $H_{kv}$ 个 KV 头共享**同一个** `RotaryPositionalEmbedding` 模块实例。所有头在每次 forward 时调用同一个 `rope.forward(seq_len)` 获取 `(cos, sin)` 表，而非每个头各自维护一份。

**Why**：
1. **节省计算**：cos/sin 表只需计算一次，各头通过广播复用，节省 $H - 1$ 次三角函数计算。
2. **缓存高效**：`_build_cache` 将表以 `register_buffer` 注册，设备迁移（`.to(device)`）自动同步。
3. **一致性**：相同位置 $t$ 在所有头上旋转角度完全一致，使多头注意力的位置依赖语义明确。

**How**（源码实现）：`RotaryPositionalEmbedding._build_cache` 预先计算长度为 `max_seq_len` 的 cos/sin 表并缓存；当序列超过缓存时自动扩容：

```python
if seq_len > self._cached_seq_len or device/dtype 不匹配:
    self._build_cache(max(seq_len, cached), device, dtype)
return self.cos_cached[:seq_len], self.sin_cached[:seq_len]  # 切片，O(1)
```

WalkieAttention 中的调用：
```python
cos, sin = self.rope(T, device=q.device, dtype=q.dtype)  # 共享，所有头共用
q = apply_rope(q, cos, sin)   # q: (B, H_q, T, d_h)
k = apply_rope(k, cos, sin)   # k: (B, H_kv, T, d_h)
```

`_build_cache` 中 `freqs` 拼接两次的设计：
```python
freqs = einsum("t,d->td", t, inv_freq)  # (T, d/2)
emb   = cat((freqs, freqs), dim=-1)     # (T, d)  前半与后半同频
```
这样 `apply_rope` 中可以直接 `cos[..., :half]` 取前半作用于 $x_1$，`cos[..., half:]` 取后半（与前半相同）作用于 $x_2$，避免额外的切片操作，代码等价于二维旋转矩阵的批量展开。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.position.rope import RotaryPositionalEmbedding, apply_rope, compute_rope_freqs

In [ ]:
import matplotlib.pyplot as plt
import torch

# 可视化不同维度 i 的频率 ω_i = θ^(-2i/d)，展示低频到高频的分布
head_dim = 64
base = 1e6
half = head_dim // 2
idx = torch.arange(half, dtype=torch.float32)
freqs_vis = 1.0 / (base ** (2.0 * idx / head_dim))

plt.figure(figsize=(9, 4))
plt.semilogy(idx.numpy(), freqs_vis.numpy(), marker='o', markersize=4)
plt.xlabel("维度索引 $i$ (0 → d/2-1)")
plt.ylabel("频率 $ω_i$（对数轴）")
plt.title(f"RoPE 频率分布 (head_dim={head_dim}, base={base:.0e})")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"最低频: {freqs_vis[0].item():.2e}  |  最高频: {freqs_vis[-1].item():.2e}")

### 3. 频率向量与 cos/sin 缓存验证

In [ ]:
freqs = compute_rope_freqs(head_dim=16, base=1e6)
print("freqs.shape:", freqs.shape)  # 应为 (8,)，即 head_dim//2
print("freqs[:4]:  ", freqs[:4].numpy())  # 低频端（接近 1.0）

# 构造缓存并验证形状
rope = RotaryPositionalEmbedding(head_dim=16, max_seq_len=8, base=1e6)
cos, sin = rope(seq_len=12, dtype=torch.float32)
print("cos.shape:", cos.shape)   # (12, 16) 按需扩容
print("sin.shape:", sin.shape)

### 4. 旋转保范性验证（二维配对范数不变）

旋转矩阵是正交矩阵，因此 $\|[x_{2i}', x_{2i+1}']^T\|_2 = \|[x_{2i}, x_{2i+1}]^T\|_2$。
此处验证 `apply_rope` 对每对相邻分量的模长（即 $x_{2i}^2 + x_{2i+1}^2$）保持不变。

In [ ]:
torch.manual_seed(0)
# 构造 (Batch=2, Heads=4, Seq=12, HeadDim=16) 的 Q 或 K 特征
x = torch.randn(2, 4, 12, 16)
cos, sin = rope(seq_len=12, dtype=x.dtype)
y = apply_rope(x, cos, sin)

# 验证每对 (x_i, x_{i+d/2}) 的二维模长旋转后不变
half = x.size(-1) // 2
# 原始每对配对的平方和：x[:d/2]^2 + x[d/2:]^2
before = x[..., :half].pow(2) + x[..., half:].pow(2)
after  = y[..., :half].pow(2) + y[..., half:].pow(2)

print("output shape:", tuple(y.shape))
print("max norm diff:", (before - after).abs().max().item())  # 应接近 0
assert (before - after).abs().max().item() < 1e-5, "旋转后模长出现误差！"

### 5. 源码精讲

```python
def compute_rope_freqs(head_dim: int, base: float = 1e6) -> torch.Tensor:
    """返回长度 head_dim // 2 的频率向量（fp32）。"""
    half = head_dim // 2
    idx = torch.arange(half, dtype=torch.float32)
    # ω_i = base^(-2i / head_dim)，即低 i 对应低频（慢变化）
    return 1.0 / (base ** (2.0 * idx / head_dim))


def apply_rope(x, cos, sin):
    """对 x 的最后一维做旋转，x: (B, H, T, D)，cos/sin: (T, D)。"""
    # 广播 cos/sin 到 (1, 1, T, D) 以匹配 x 的批量维度
    while cos.dim() < x.dim():
        cos = cos.unsqueeze(0)
        sin = sin.unsqueeze(0)
    half = x.size(-1) // 2
    x1, x2 = x.chunk(2, dim=-1)      # 各取前 d/2 和后 d/2
    cos_half = cos[..., :half]
    sin_half = sin[..., :half]
    # 对应公式：x_rot = x1*cos - x2*sin  ‖  x2*cos + x1*sin
    return torch.cat(
        (x1 * cos_half - x2 * sin_half,
         x2 * cos_half + x1 * sin_half),
        dim=-1,
    )
```

`RotaryPositionalEmbedding._build_cache` 中：
```python
freqs = torch.einsum("t,d->td", t, self.inv_freq)  # (T, d/2)
emb   = torch.cat((freqs, freqs), dim=-1)           # (T, d) 拼两次使后半与前半同频
cos   = emb.cos()  # cos 表
sin   = emb.sin()  # sin 表
```
拼接两次的作用是令 `cos_half` / `sin_half` 能直接与 `x1`/`x2` 的形状对齐，避免额外的切片操作。

---

## 延伸阅读与参考资料

### 核心论文
- **RoFormer: Enhanced Transformer with Rotary Position Embedding**: Su et al., 2021. [arXiv:2104.09864](https://arxiv.org/abs/2104.09864)
- **LLaMA**: Touvron et al., 2023. [arXiv:2302.13971](https://arxiv.org/abs/2302.13971)

### 工程实现与博客
- **Hugging Face RoPE utilities**: [source](https://github.com/huggingface/transformers/blob/main/src/transformers/modeling_rope_utils.py)
- **RoPE 原理推导（苏剑林）**: [kexue.fm](https://kexue.fm/archives/8265)